# Pivot Tables

*Summarize revenue across two categorical dimensions and verify the margins.*


## Before You Run

In **Add files**, select `retail_sales_clean.csv` from this lesson's supplied assets.
Enter `assets/08_pivot_tables` for **Save folder**. The upload creates
missing folders and places the file at
`/home/pyodide/assets/08_pivot_tables/retail_sales_clean.csv`.

`/home/pyodide` is the browser's virtual filesystem, not a folder on your
computer. The upload folder is relative to `/home/pyodide`; Python relative
paths use the current directory returned by `os.getcwd()`. Keep the default
working directory in the course site. Locally, run from this chapter folder.
Uploading again to the same complete path replaces that input file.
Saved examples appear under `outputs/08_pivot_tables/` in the file browser.


## Prepare a Numeric Measure

A pivot table needs a measure to summarize. Multiply quantity by unit
price to obtain order revenue in dollars. This supplied 15-row exploration
file is independent of the raw-data cleaning example and includes extra
orders; it is not that example's saved output.


In [1]:
import os

import pandas as pd

input_directory = os.path.join("assets", "08_pivot_tables")
output_directory = os.path.join("outputs", "08_pivot_tables")
input_path = os.path.join(input_directory, "retail_sales_clean.csv")
output_path = os.path.join(output_directory, "revenue_by_region_and_category.csv")
os.makedirs(output_directory, exist_ok=True)

sales_df = pd.read_csv(input_path, encoding="utf-8")
sales_df["revenue"] = sales_df["quantity"] * sales_df["unit_price"]

print("Revenue records (dollars):")
print(sales_df[["order_id", "region", "category", "quantity", "unit_price", "revenue"]])


Revenue records (dollars):
    order_id   region   category  quantity  unit_price  revenue
0       1001    Seoul     Laptop         2        1200     2400
1       1002    Busan  Accessory         5          25      125
2       1003    Seoul    Monitor         1         310      310
3       1004  Incheon     Laptop         1         980      980
4       1005    Daegu  Accessory         4          45      180
5       1006    Seoul  Accessory         3          25       75
6       1007    Busan    Monitor         2         310      620
7       1008  Incheon     Laptop         1        1200     1200
8       1009    Daegu  Accessory         2         140      280
9       1010    Seoul     Laptop         2         980     1960
10      1011    Busan  Accessory         3          45      135
11      1012  Incheon    Monitor         1         310      310
12      1013    Seoul  Accessory        10          25      250
13      1014    Daegu     Laptop         1        1200     1200
14      1015 

Each row now has a revenue value. Order 1001 contributes 2 × 1,200 = 2,400 dollars to the Seoul–Laptop combination.


## Assign Row Keys, Column Keys, and an Aggregation

| Argument | Meaning in this example |
|---|---|
| `values="revenue"` | Measure to summarize |
| `index="region"` | Categories on the row axis |
| `columns="category"` | Categories on the column axis |
| `aggfunc="sum"` | Add order revenue in each combination |
| `fill_value=0` | Display absent combinations as zero |
| `margins=True` | Add `All` row and column totals |

Specify `aggfunc` explicitly; the default is a mean, not a total.


In [2]:
revenue_pivot = pd.pivot_table(
    sales_df,
    values="revenue",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)

print("Revenue by region and category (dollars):")
print(revenue_pivot)
print("Order revenue total:", sales_df["revenue"].sum())
print("Pivot grand total:", revenue_pivot.loc["All", "All"])
print("Grand totals agree:", revenue_pivot.loc["All", "All"] == sales_df["revenue"].sum())


Revenue by region and category (dollars):
category  Accessory  Laptop  Monitor    All
region                                     
Busan           400       0      620   1020
Daegu           460    1200        0   1660
Incheon           0    2180      310   2490
Seoul           325    4360      310   4995
All            1185    7740     1240  10165
Order revenue total: 10165
Pivot grand total: 10165
Grand totals agree: True


Both totals are 10,165 dollars. Seoul contributes 4,995, the largest regional total. Laptop revenue is 7,740. Daegu–Monitor is zero because this dataset contains no orders in that combination; it does not indicate a free monitor.


## Preserve Row Labels in the Saved Report

The region labels are currently the pivot's index. `.reset_index()` moves
them into a column before saving with `index=False`. The totals are already
included, so summing the entire pivot would count the same revenue repeatedly.


In [3]:
pivot_report = revenue_pivot.reset_index()
pivot_report.to_csv(output_path, index=False, encoding="utf-8")
reopened_df = pd.read_csv(output_path, encoding="utf-8")

print("Saved file:", output_path)
print("Reopened pivot report:")
print(reopened_df)
print("Round trip matches:", reopened_df.equals(pivot_report))


Saved file: outputs/08_pivot_tables/revenue_by_region_and_category.csv
Reopened pivot report:
    region  Accessory  Laptop  Monitor    All
0    Busan        400       0      620   1020
1    Daegu        460    1200        0   1660
2  Incheon          0    2180      310   2490
3    Seoul        325    4360      310   4995
4      All       1185    7740     1240  10165
Round trip matches: True


The saved table has four region rows plus an `All` row. Its five columns include the explicit region label, three product categories, and the row total. The round trip matches.


Choose the measure, row keys, column keys, and aggregation deliberately. Check the grand total against the original records, and keep pivot labels when exporting.

[Pivot table arguments](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
